In [1]:
import os
print("Danh sách thư mục dữ liệu:")
print(os.listdir('/kaggle/input/'))

Danh sách thư mục dữ liệu:
['datasets']


In [5]:
import os
import glob
import numpy as np
import pandas as pd
from scipy.signal import resample
from tqdm.notebook import tqdm

# 1. Quét tất cả file CSV trong Kaggle
csv_files = sorted(glob.glob("/kaggle/input/**/*.csv", recursive=True))
print(f"-> Đã tìm thấy {len(csv_files)} file CSV.")

# 2. Định nghĩa hàm xử lý: Resample 50Hz + Cắt cửa sổ trượt [100, 6]
def process_kfall_df(
    df, 
    original_fs=100,      # Tần số gốc K-Fall là 100Hz
    target_fs=50,         # Tần số mục tiêu 50Hz
    window_size_sec=2.0,  # Cửa sổ 2 giây -> 100 mẫu
    overlap_ratio=0.5     # Trượt 50%
):
    acc_cols = ['AccX', 'AccY', 'AccZ']
    gyr_cols = ['GyrX', 'GyrY', 'GyrZ']
    
    feature_cols = [c for c in acc_cols + gyr_cols if c in df.columns]
    
    if len(feature_cols) < 6:
        return None

    data = df[feature_cols].to_numpy(dtype=np.float32)
    
    # Resample về 50Hz
    num_original_samples = len(data)
    num_target_samples = int(num_original_samples * target_fs / original_fs)
    resampled_data = resample(data, num_target_samples, axis=0)
    
    # Cắt cửa sổ trượt [100, 6]
    window_length = int(window_size_sec * target_fs)   # 100 mẫu
    step_size = int(window_length * (1.0 - overlap_ratio)) # Nấc trượt 50 mẫu
    
    windows = []
    start = 0
    total_samples = len(resampled_data)
    
    while start + window_length <= total_samples:
        win = resampled_data[start : start + window_length]
        windows.append(win)
        start += step_size
        
    if len(windows) == 0:
        return None
        
    return np.array(windows, dtype=np.float32)

# 3. Đọc dữ liệu và hiển thị thanh tiến trình (Progress Bar)
all_windows = []
print("Đang tiến hành tiền xử lý và cắt cửa sổ trượt...")

for f_path in tqdm(csv_files):
    try:
        df = pd.read_csv(f_path)
        wins = process_kfall_df(df, original_fs=100, target_fs=50)
        if wins is not None:
            all_windows.append(wins)
    except Exception:
        continue

if len(all_windows) > 0:
    X_data = np.vstack(all_windows)
    print(f"\n==========================================")
    print(f"=> XỬ LÝ THÀNH CÔNG TOÀN BỘ DỮ LIỆU!")
    print(f"-> Tổng số cửa sổ thu được: {X_data.shape[0]}")
    print(f"-> Kích thước mỗi cửa sổ: {X_data[0].shape} (100 frame x 6 trục IMU)")
    print(f"==========================================")
else:
    print("\n[!] Vui lòng kiểm tra lại tên các cột trong file CSV!")

-> Đã tìm thấy 5075 file CSV.
Đang tiến hành tiền xử lý và cắt cửa sổ trượt...


  0%|          | 0/5075 [00:00<?, ?it/s]


=> XỬ LÝ THÀNH CÔNG TOÀN BỘ DỮ LIỆU!
-> Tổng số cửa sổ thu được: 32365
-> Kích thước mỗi cửa sổ: (100, 6) (100 frame x 6 trục IMU)


In [6]:
# Cell 3: Trích xuất nhãn (y_data) tương ứng với các cửa sổ X_data
# (Ví dụ: 0 = Normal, 1 = Fall dựa trên tên file hoặc nhãn K-Fall)

print(f"Kích thước X_data: {X_data.shape}")
# X_data hiện đã sẵn sàng để đưa vào StandardScaler và mô hình 1D-CNN / LSTM!

Kích thước X_data: (32365, 100, 6)


In [7]:
import numpy as np

# Lưu dạng nén .npz giúp giảm dung lượng file
np.savez_compressed("X_kfall_50Hz_100x6.npz", X_data=X_data)
print("-> Đã lưu file nén X_kfall_50Hz_100x6.npz thành công!")

-> Đã lưu file nén X_kfall_50Hz_100x6.npz thành công!
